# Customer Churn Prediction — Step 2: Data Quality Audit

**Dataset:** Telco Customer Churn (Kaggle / IBM Watson Analytics)  
**Goal:** Systematically inspect the raw dataset for quality issues — missing values, wrong data types, duplicates, suspicious values, and anything that will need attention in Step 3 (Data Cleaning).

> ⚠️ **Nothing is modified in this notebook.** The raw CSV is read once and only inspected. Every cell is purely observational.

---

## 1. Setup — Import Libraries and Load Data

We need:
- **`pandas`** — for loading the CSV and performing all tabular inspections.
- **`numpy`** — to detect special numeric sentinels like `NaN` and `inf`.
- **`os`** — to build a cross-platform file path to the dataset.

`pd.set_option` widens the display so that DataFrames with many columns print without truncation.

In [1]:
import os
import numpy as np
import pandas as pd

# Show all columns and a reasonable number of rows without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

# ── Resolve dataset path ────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == 'notebook' else NOTEBOOK_DIR
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('Dataset path:', DATA_PATH)
print('File exists :', os.path.exists(DATA_PATH))

Dataset path: c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\WA_Fn-UseC_-Telco-Customer-Churn.csv
File exists : True


In [2]:
# Load the raw CSV — no type coercions, no parsing tricks, data as-is
df = pd.read_csv(DATA_PATH)
print('Loaded successfully.')

Loaded successfully.


---
## 2. Dataset Shape

The very first check: how many rows (customer records) and columns (features) do we have?  
This gives us the total counts we will reference throughout every other check — for example, when computing missing-value percentages.

In [3]:
rows, cols = df.shape
print(f'Rows    (customer records) : {rows:,}')
print(f'Columns (features)         : {cols}')

Rows    (customer records) : 7,043
Columns (features)         : 21


---
## 3. Duplicate Rows

`df.duplicated()` returns a boolean Series: `True` for every row that is an exact copy of an earlier row.  
`.sum()` counts how many duplicates exist.  
If duplicates are found, we display them so we can examine them closely before deciding what to do in Step 3.

In [4]:
n_duplicates = df.duplicated().sum()
print(f'Duplicate rows found: {n_duplicates}')

if n_duplicates > 0:
    print('\nDuplicate rows:')
    display(df[df.duplicated(keep=False)])
else:
    print('No duplicate rows — ✓')

Duplicate rows found: 0
No duplicate rows — ✓


---
## 4. Missing Values (Absolute Count)

`df.isnull()` produces a boolean DataFrame: `True` wherever a value is `NaN` (or `None`).  
`.sum()` collapses it column-by-column to give the count of missing values per column.  

We then filter to only the columns that actually have at least one missing value, so the output stays clean.  
If nothing is found, that does **not** mean the data is clean — blank strings stored as `' '` will not be caught here (that is handled in Check 12).

In [5]:
missing_counts = df.isnull().sum()
missing_cols   = missing_counts[missing_counts > 0]

if missing_cols.empty:
    print('No NaN/None missing values detected by pandas. \n'
          '(Note: blank strings are NOT counted here — see Check 12 for TotalCharges.)')
else:
    print('Columns with missing values (NaN/None):')
    print(missing_cols.to_string())

No NaN/None missing values detected by pandas. 
(Note: blank strings are NOT counted here — see Check 12 for TotalCharges.)


---
## 5. Missing-Value Percentage per Column

Raw counts are hard to interpret on a dataset of varying size.  
Here we compute the **percentage** of missing values for every column: `(missing count / total rows) × 100`.  
This makes it easy to spot columns with a high fraction of missing data, which may require dropping rather than imputing in Step 3.

In [6]:
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count'      : df.isnull().sum(),
    'Missing Percentage' : missing_pct
}).sort_values('Missing Percentage', ascending=False)

# Show only columns that have at least one missing value
print('Columns with at least one missing value:')
has_missing = missing_summary[missing_summary['Missing Count'] > 0]

if has_missing.empty:
    print('None found (for NaN/None — blank strings checked separately).')
else:
    display(has_missing)

Columns with at least one missing value:
None found (for NaN/None — blank strings checked separately).


---
## 6. Data Types of Every Column

`df.dtypes` returns the pandas-inferred data type for each column.  

Why does this matter?  
- A column that **should** be numeric (like `TotalCharges`) but is stored as `object` is a red flag — it means there are non-numeric characters hiding in that column.
- A column that is `int64` when it should be `float64` may lose precision.
- `object` columns are pandas' default for strings, but also for mixed types.

We display each column alongside its dtype and whether it is numeric or not.

In [7]:
dtype_df = pd.DataFrame({
    'Data Type'   : df.dtypes,
    'Is Numeric'  : df.dtypes.apply(lambda dt: pd.api.types.is_numeric_dtype(dt))
})

print(f'Total columns: {len(dtype_df)}')
display(dtype_df)

Total columns: 21


,Data Type,Is Numeric
customerID,object,False
gender,object,False
SeniorCitizen,int64,True
Partner,object,False
Dependents,object,False
tenure,int64,True
PhoneService,object,False
MultipleLines,object,False
InternetService,object,False
OnlineSecurity,object,False


---
## 7. Unique Value Counts per Column

`df.nunique()` counts the number of **distinct** values in each column.  

This is useful for two things:
1. **Identifying categorical columns** — columns with a small number of unique values (e.g. 2–10) are almost certainly categorical.
2. **Spotting ID-like columns** — a column where every value is unique (nunique ≈ row count) is likely an identifier, not a feature.

We rank columns from fewest to most unique values.

In [8]:
unique_counts = df.nunique().sort_values()

unique_df = pd.DataFrame({
    'Unique Values' : unique_counts,
    'Likely Type'   : unique_counts.apply(
        lambda n: 'Binary'      if n == 2
             else 'Categorical' if n <= 15
             else 'Numerical / ID'
    )
})

display(unique_df)

,Unique Values,Likely Type
Churn,2,Binary
gender,2,Binary
SeniorCitizen,2,Binary
Partner,2,Binary
Dependents,2,Binary
PaperlessBilling,2,Binary
PhoneService,2,Binary
Contract,3,Categorical
StreamingMovies,3,Categorical
StreamingTV,3,Categorical


---
## 8. Unique Values for Categorical Columns

For every column that has **15 or fewer unique values**, we print the actual values.  
This lets us:
- Spot inconsistent capitalisation (e.g. `'Yes'` vs `'yes'`).
- Find unexpected placeholder strings like `'No internet service'` or `'No phone service'` that might need harmonising.
- Confirm the target variable `Churn` only contains valid labels.

We exclude `customerID` explicitly since it is a unique identifier, not a feature.

In [9]:
CATEGORICAL_THRESHOLD = 15

cat_cols = [col for col in df.columns
            if df[col].nunique() <= CATEGORICAL_THRESHOLD
            and col != 'customerID']

print(f'Columns with ≤ {CATEGORICAL_THRESHOLD} unique values (excluding customerID):\n')
for col in cat_cols:
    vals = sorted(df[col].dropna().unique().tolist())
    print(f'  {col:<30} ({df[col].nunique()} unique) → {vals}')

Columns with ≤ 15 unique values (excluding customerID):

  gender                         (2 unique) → ['Female', 'Male']
  SeniorCitizen                  (2 unique) → [0, 1]
  Partner                        (2 unique) → ['No', 'Yes']
  Dependents                     (2 unique) → ['No', 'Yes']
  PhoneService                   (2 unique) → ['No', 'Yes']
  MultipleLines                  (3 unique) → ['No', 'No phone service', 'Yes']
  InternetService                (3 unique) → ['DSL', 'Fiber optic', 'No']
  OnlineSecurity                 (3 unique) → ['No', 'No internet service', 'Yes']
  OnlineBackup                   (3 unique) → ['No', 'No internet service', 'Yes']
  DeviceProtection               (3 unique) → ['No', 'No internet service', 'Yes']
  TechSupport                    (3 unique) → ['No', 'No internet service', 'Yes']
  StreamingTV                    (3 unique) → ['No', 'No internet service', 'Yes']
  StreamingMovies                (3 unique) → ['No', 'No internet service',

---
## 9. Numerical Summary Statistics

`df.describe()` computes standard descriptive statistics for every numeric column:  
count, mean, standard deviation, min, 25th percentile (Q1), median (Q2 / 50th), 75th percentile (Q3), and max.

What to look for:
- **`min` is negative** for a column that should never be negative (e.g. charges, tenure).
- **`max` is astronomically large** — potential outlier or data entry error.
- **`count` < total rows** — missing values in that numeric column.
- **`std = 0`** — the column has no variance (constant, useless as a feature).

Note: `TotalCharges` will likely **not** appear here because it was read as `object` — which is itself a finding.

In [10]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns detected by pandas: {num_cols}\n')

display(df[num_cols].describe().T)

Numeric columns detected by pandas: ['SeniorCitizen', 'tenure', 'MonthlyCharges']



,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


---
## 10. Target Variable — `Churn` Audit

The target variable is the column we are trying to predict: **`Churn`**.  
Before building any model, we must confirm:
1. It only contains the expected labels (`'Yes'` and `'No'`).
2. There are no `NaN` values.
3. The class distribution is not severely imbalanced (highly imbalanced targets require special handling in modelling).

`value_counts(dropna=False)` counts every value including `NaN` so nothing is hidden.

In [11]:
target_col = 'Churn'

print('=== Target Variable Audit: Churn ===')
print(f'  Data type   : {df[target_col].dtype}')
print(f'  Null count  : {df[target_col].isnull().sum()}')
print(f'  Unique vals : {df[target_col].unique().tolist()}\n')

churn_counts = df[target_col].value_counts(dropna=False)
churn_pct    = df[target_col].value_counts(dropna=False, normalize=True).mul(100).round(2)

churn_summary = pd.DataFrame({
    'Count'      : churn_counts,
    'Percentage' : churn_pct
})

print('Value distribution:')
display(churn_summary)

# Check for unexpected values
expected_values = {'Yes', 'No'}
actual_values   = set(df[target_col].dropna().unique())
unexpected      = actual_values - expected_values

if unexpected:
    print(f'\n⚠️  Unexpected values in Churn column: {unexpected}')
else:
    print('\nNo unexpected values in Churn column. ✓')

=== Target Variable Audit: Churn ===
  Data type   : object
  Null count  : 0
  Unique vals : ['No', 'Yes']

Value distribution:


,Count,Percentage
Churn,,
No,5174,73.46
Yes,1869,26.54



No unexpected values in Churn column. ✓


---
## 11. Suspicious / Impossible Values in Numerical Columns

Even when pandas reports no `NaN` values, a numeric column can still contain problematic data:
- **Negative values** in columns that should be non-negative (`tenure`, `MonthlyCharges`, `TotalCharges`).
- **Infinite values** (`np.inf`, `-np.inf`) that cause silent errors in ML models.
- **Zero values** in columns where zero is physically impossible (e.g. a customer with 0 monthly charge but active service).

We check each numeric column for all three.

In [12]:
print('=== Suspicious Value Check — Numeric Columns ===\n')

findings = []

for col in num_cols:
    series = df[col]
    n_neg    = (series < 0).sum()
    n_inf    = np.isinf(series).sum()
    n_zero   = (series == 0).sum()

    findings.append({
        'Column'        : col,
        'Negative Count': int(n_neg),
        'Infinite Count': int(n_inf),
        'Zero Count'    : int(n_zero),
        'Min'           : series.min(),
        'Max'           : series.max()
    })

findings_df = pd.DataFrame(findings).set_index('Column')
display(findings_df)

# Flag any suspicious column
suspicious = findings_df[(findings_df['Negative Count'] > 0) | (findings_df['Infinite Count'] > 0)]
if suspicious.empty:
    print('\nNo negative or infinite values found in numeric columns. ✓')
else:
    print('\n⚠️  Columns with negative or infinite values:')
    display(suspicious)

=== Suspicious Value Check — Numeric Columns ===



,Negative Count,Infinite Count,Zero Count,Min,Max
Column,,,,,
SeniorCitizen,0,0,5901,0.00,1.00
tenure,0,0,11,0.00,72.00
MonthlyCharges,0,0,0,18.25,118.75



No negative or infinite values found in numeric columns. ✓


---
## 12. Deep-Dive: `TotalCharges` Column Investigation

**Why single out `TotalCharges`?**  
This column **should** be numeric (it is a dollar amount), but if pandas read it as `object` (string), it means the column contains at least one non-numeric value.

The most common culprit in this dataset is **blank spaces (`' '`)** for customers with very short tenure — their total charge has not yet been calculated, so the field was left as an empty string in the source system instead of being recorded as `0` or `NaN`.

We will:
1. Confirm the current dtype.
2. Try to force-convert the column to numeric with `pd.to_numeric(errors='coerce')` — any value that cannot be parsed becomes `NaN`.
3. Count how many values failed conversion (those are the problematic entries).
4. Display the actual raw values that failed, so we know exactly what we are dealing with.

> The conversion result is stored in a **temporary variable only** — the original `df` is NOT modified.

In [13]:
col = 'TotalCharges'

print(f'=== Deep-Dive: {col} ===')
print(f'  Current dtype : {df[col].dtype}')
print(f'  Sample values : {df[col].head(10).tolist()}\n')

=== Deep-Dive: TotalCharges ===
  Current dtype : object
  Sample values : ['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95']



In [14]:
# Force-convert to numeric; anything that cannot be parsed becomes NaN
# We store the result in a TEMPORARY variable — df is NOT changed
tc_numeric = pd.to_numeric(df[col], errors='coerce')

# Find the rows where conversion failed (original was non-numeric)
failed_mask  = tc_numeric.isnull() & df[col].notnull()  # was not already NaN, but failed conversion
n_failed     = failed_mask.sum()

print(f'Values that could NOT be converted to numeric: {n_failed}')

if n_failed > 0:
    print('\nRaw values causing the failure (showing up to 20):')
    problem_vals = df.loc[failed_mask, [col, 'tenure', 'MonthlyCharges']].head(20)
    display(problem_vals)

    print('\nDistinct raw string values in the failing entries:')
    print(df.loc[failed_mask, col].unique().tolist())
else:
    print('All values in TotalCharges can be parsed as numeric. ✓')

Values that could NOT be converted to numeric: 11

Raw values causing the failure (showing up to 20):


,TotalCharges,tenure,MonthlyCharges
488,,0,52.55
753,,0,20.25
936,,0,80.85
1082,,0,25.75
1340,,0,56.05
3331,,0,19.85
3826,,0,25.35
4380,,0,20.00
5218,,0,19.70
6670,,0,73.35



Distinct raw string values in the failing entries:
[' ']


In [15]:
# Additional check: how many entries are blank strings or whitespace-only?
n_blank = df[col].astype(str).str.strip().eq('').sum()
print(f'Entries that are blank / whitespace-only strings: {n_blank}')

# Descriptive stats if we were to treat TotalCharges as numeric (read-only peek)
print('\nDescriptive stats (treating TotalCharges as numeric, NaN for blanks):')
display(tc_numeric.describe().to_frame(name=col).T)

Entries that are blank / whitespace-only strings: 11

Descriptive stats (treating TotalCharges as numeric, NaN for blanks):


,count,mean,std,min,25%,50%,75%,max
TotalCharges,7032.0,2283.300441,2266.771362,18.8,401.45,1397.475,3794.7375,8684.8


---
## 13. Full Column-Level Quality Summary Table

A single consolidated table across all columns showing:  
`dtype`, `non-null count`, `null count`, `null %`, `unique count`, and `sample values`.  

This is the all-in-one reference table for the findings section below.

In [16]:
summary_rows = []

for col in df.columns:
    null_count = df[col].isnull().sum()
    null_pct   = round(null_count / len(df) * 100, 2)
    n_unique   = df[col].nunique()
    sample     = df[col].dropna().unique()[:4].tolist()

    summary_rows.append({
        'Column'       : col,
        'Dtype'        : str(df[col].dtype),
        'Non-Null'     : len(df) - null_count,
        'Null Count'   : null_count,
        'Null %'       : null_pct,
        'Unique Values': n_unique,
        'Sample Values': sample
    })

quality_table = pd.DataFrame(summary_rows).set_index('Column')
display(quality_table)

,Dtype,Non-Null,Null Count,Null %,Unique Values,Sample Values
Column,,,,,,
customerID,object,7043,0,0.0,7043,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOCW]"
gender,object,7043,0,0.0,2,"[Female, Male]"
SeniorCitizen,int64,7043,0,0.0,2,"[0, 1]"
Partner,object,7043,0,0.0,2,"[Yes, No]"
Dependents,object,7043,0,0.0,2,"[No, Yes]"
tenure,int64,7043,0,0.0,73,"[1, 34, 2, 45]"
PhoneService,object,7043,0,0.0,2,"[No, Yes]"
MultipleLines,object,7043,0,0.0,3,"[No phone service, No, Yes]"
InternetService,object,7043,0,0.0,3,"[DSL, Fiber optic, No]"


---

# 📋 Data Quality Findings

This section summarises every issue discovered in the checks above.  
**Nothing has been fixed yet.** These findings are the input specification for Step 3: Data Cleaning.

---

### ✅ What Looks Fine
| Item | Status |
|---|---|
| Duplicate rows | None detected |
| Target variable `Churn` unique values | Only `'Yes'` and `'No'` — no unexpected labels |
| `Churn` null values | None |
| Negative values in numeric columns | None in `tenure`, `MonthlyCharges` |
| Infinite values | None detected |

---

### ⚠️ Issues Found — To Be Fixed in Step 3

#### 1. `TotalCharges` — Wrong Data Type
- **Current dtype:** `object` (string)
- **Expected dtype:** `float64` (numeric)
- **Root cause:** Some entries contain a blank string `' '` (whitespace) instead of a numeric value. These are customers with `tenure = 0` who joined but were never charged. Pandas cannot auto-convert the column because of these entries, so it reads the entire column as `object`.
- **Action needed in Step 3:** Replace blank/whitespace entries with `NaN`, then cast the column to `float64`. Decide on an imputation strategy (e.g. fill with `0` or with `MonthlyCharges` for tenure-0 customers).

#### 2. `TotalCharges` — Hidden Missing Values
- **pandas `isnull()` count:** 0 (because blanks are stored as strings, not `NaN`)
- **True missing count:** 11 entries that are whitespace-only strings
- **Action needed in Step 3:** Treat these as missing and handle via imputation or row removal.

#### 3. Class Imbalance in `Churn` (Target Variable)
- The dataset is imbalanced: roughly **73% No** and **27% Yes**.
- This is not a data quality error but it is an important finding. ML models trained on imbalanced data tend to be biased toward the majority class.
- **Action needed in Step 6 (Modelling):** Use techniques such as SMOTE, class-weight balancing, or stratified cross-validation.

#### 4. `SeniorCitizen` — Stored as `int64` Instead of Categorical
- **Current dtype:** `int64` with only values `0` and `1`.
- **Expected:** Binary categorical (`No` / `Yes`) — consistent with all other binary columns in the dataset.
- **Action needed in Step 3:** Convert to `object` or a consistent binary label (e.g. `'Yes'`/`'No'`) to match the encoding of other binary columns.

#### 5. Redundant / Placeholder Values in Service Columns
- Several columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) contain a third value: **`'No internet service'`** in addition to `'Yes'` and `'No'`.
- Similarly, `MultipleLines` contains **`'No phone service'`**.
- These are not truly distinct categories — `'No internet service'` is equivalent to `'No'` for the purpose of those features, since the underlying condition (internet service) is already captured in the `InternetService` column.
- **Action needed in Step 3 / Step 5 (Feature Engineering):** Decide whether to collapse `'No internet service'` → `'No'` and `'No phone service'` → `'No'`, or keep them as a separate third category.

#### 6. `customerID` — Identifier Column, Not a Feature
- `customerID` has one unique value per row — it is a row identifier with no predictive value.
- **Action needed in Step 3:** Drop this column before any analysis or modelling.

---

### 📌 Summary Checklist for Step 3 (Data Cleaning)

| Priority | Column(s) | Issue | Action |
|---|---|---|---|
| 🔴 High | `TotalCharges` | Stored as string; blank entries for 11 rows | Strip, cast to `float64`, impute blanks |
| 🔴 High | `customerID` | Identifier, not a feature | Drop column |
| 🟡 Medium | `SeniorCitizen` | `int64` (0/1) instead of categorical label | Convert to `'Yes'`/`'No'` |
| 🟡 Medium | `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `MultipleLines` | Extra placeholder values (`'No internet service'`, `'No phone service'`) | Review in Step 5 — consider collapsing to `'No'` |
| 🟢 Low | `Churn` | Class imbalance (~73/27) | Address during modelling in Step 6 |

---

**Next step →** Step 3: Data Cleaning — apply all the fixes listed above to produce a clean DataFrame ready for EDA.